# Sleep Cannabis Alcohol Study

- Merging merged actigraphy data from the 4 studies with the scored (scrubbed) actigraphy data from each study

Notes: 
- Use scored data for start and stop times.
- Use the actigraphy data for determining sleep metrics
- Calculate sleep biomarkers for: 
  1. sleep onset to sleep offset (lets me verify data outputs compared to orignal scored dataset)
  2. First 1/2 of sleep night i.e., start sleep to mid-sleep)
  3. first 4-h of sleep (sleep start to sleep start + 4h)

## Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

#### Actigraphy Data

In [2]:
actigraphy_df = pd.read_csv('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Data_analysis/full_actigraphy_df_cleaned.csv', index_col=0, dtype={'date':str, 'time':str})

actigraphy_df.head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
0,DXA_174,DXA,3/27/2025,12:49:00 PM,769.0,2025-03-27 12:49:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
1,DXA_174,DXA,3/27/2025,12:50:00 PM,770.0,2025-03-27 12:50:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
2,DXA_174,DXA,3/27/2025,12:51:00 PM,771.0,2025-03-27 12:51:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
3,DXA_174,DXA,3/27/2025,12:52:00 PM,772.0,2025-03-27 12:52:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
4,DXA_174,DXA,3/27/2025,12:53:00 PM,773.0,2025-03-27 12:53:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah


In [3]:
## convert dtypes for analysis
actigraphy_df['subject_id'] = actigraphy_df['subject_id'].astype(str)
actigraphy_df['subject_id'] = actigraphy_df['subject_id'].replace('SNA1', 'SNA').replace('SPU1', 'SPU')
actigraphy_df['date_time24'] = pd.to_datetime(actigraphy_df['date'] + ' ' + actigraphy_df['time'], format='mixed')

actigraphy_df.head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
0,DXA_174,DXA,3/27/2025,12:49:00 PM,769.0,2025-03-27 12:49:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
1,DXA_174,DXA,3/27/2025,12:50:00 PM,770.0,2025-03-27 12:50:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
2,DXA_174,DXA,3/27/2025,12:51:00 PM,771.0,2025-03-27 12:51:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
3,DXA_174,DXA,3/27/2025,12:52:00 PM,772.0,2025-03-27 12:52:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah
4,DXA_174,DXA,3/27/2025,12:53:00 PM,773.0,2025-03-27 12:53:00,0.0,NaN,0.0,NaN,ACTIVE,dxa174_3_27_2025_12_49_00_pm_174_24hr_ah


In [4]:
actigraphy_df[actigraphy_df['subject_id']=='DXA_001'].head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
903456,DXA_001,DXA,9/20/19,2:19:00 PM,NaN,2019-09-20 14:19:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903457,DXA_001,DXA,9/20/19,2:20:00 PM,NaN,2019-09-20 14:20:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903458,DXA_001,DXA,9/20/19,2:21:00 PM,NaN,2019-09-20 14:21:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903459,DXA_001,DXA,9/20/19,2:22:00 PM,NaN,2019-09-20 14:22:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903460,DXA_001,DXA,9/20/19,2:23:00 PM,NaN,2019-09-20 14:23:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined


In [5]:
actigraphy_df.dtypes

subject_id                  object
study                       object
date                        object
time                        object
time_epoch                 float64
date_time24         datetime64[ns]
off_wrist_status           float64
activity                   float64
marker                     float64
sleep/wake                 float64
interval_status             object
participant                 object
dtype: object

In [6]:
actigraphy_df[actigraphy_df['subject_id']=='DXA_001'].head()

,subject_id,study,date,time,time_epoch,date_time24,off_wrist_status,activity,marker,sleep/wake,interval_status,participant
903456,DXA_001,DXA,9/20/19,2:19:00 PM,NaN,2019-09-20 14:19:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903457,DXA_001,DXA,9/20/19,2:20:00 PM,NaN,2019-09-20 14:20:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903458,DXA_001,DXA,9/20/19,2:21:00 PM,NaN,2019-09-20 14:21:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903459,DXA_001,DXA,9/20/19,2:22:00 PM,NaN,2019-09-20 14:22:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined
903460,DXA_001,DXA,9/20/19,2:23:00 PM,NaN,2019-09-20 14:23:00,0.0,NaN,0.0,NaN,ACTIVE,dxa001_9_20_2019_2_19_00_pm_cc_24hr_combined


#### Scored Sleep Data

In [7]:
scored_sleep_df = pd.read_excel('/Users/thomasgooding/Desktop/Rutgers_Sleep_Cannabis_Alcohol_study/Data_analysis/sleep_merged_df_cleaned_062326.xlsx', index_col=0)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].astype(str)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].astype(str)
scored_sleep_df['subject_id'] = scored_sleep_df['subject_id'].str.replace('SNA1', 'SNA').str.replace('SPU1', 'SPU')

dt_columns = ['start_date', 'start_time', 'end_date',
       'end_time', 'start_datetime', 'end_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h']

for col in dt_columns:
    scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])

scored_sleep_df.head()

/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_79642/3161584176.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])
/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_79642/3161584176.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scored_sleep_df[col] = pd.to_datetime(scored_sleep_df[col])


,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-07-07 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
1,DXA_001,DXA001,Sleep,2,Saturday,NaT,NaN,NaT,NaT,NaN,...,NaN,EXCLUDED - >15% of the day off-wrist,DXA,NaN,NaT,NaT,NaN,NaN,NaT,NaT
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-07-07 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-07-07 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-07-07 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00


## Data Cleaning

In [8]:
actigraphy_IDS = list(actigraphy_df['subject_id'].unique())

scored_IDs = list(scored_sleep_df['subject_id'].unique())

In [9]:
missing_in_scored = set(actigraphy_IDS) - set(scored_IDs)

print(f'Participants in Actigraphy but not in scored dataset:')
print(f'{missing_in_scored}')

Participants in Actigraphy but not in scored dataset:
set()


In [10]:
missing_in_actigraphy = set(scored_IDs) - set(actigraphy_IDS)

print('Participants in scored dataset not found in the actigraphy_files_df:' )
print(f'{missing_in_actigraphy}')

Participants in scored dataset not found in the actigraphy_files_df:
{'DXA_109', 'DXA_052', 'DXA_105', 'DXA_048', 'DXA_003', 'DXA_005', 'DXA_128'}


## Filter participants to drop

In [11]:
## drop these participants based on their status from completing the study (i.e., withdrew or D/c for one reason or another)
pid_to_drop = list(missing_in_actigraphy)

## Drop excluded nights of sleep 

In [12]:
## Filter out flagged nights from scored_sleep_df
print("flag_actigraph value counts:")
print(scored_sleep_df['flag_actigraph'].value_counts(dropna=False))

flag_actigraph value counts:
flag_actigraph
NaN                                                                                                                                                                                  2585
EXCLUDED - >15% of the day off-wrist                                                                                                                                                  142
Times adjsuted 1 hour behind for DST                                                                                                                                                  105
DST - adjusted sleep onset and offset to 1 hr back                                                                                                                                     74
OFF-WRIST - cannot determine sleep onset or offset                                                                                                                                     38
ALL-NIGHTER               

In [13]:
## keep only nights where flag_actigraph is null/empty (not flagged)
scored_sleep_filtered_df = scored_sleep_df[scored_sleep_df['flag_actigraph'].isna()].copy()

print(f"Total nights before flag filter : {len(scored_sleep_df)}")
print(f"Total nights after flag filter  : {len(scored_sleep_filtered_df)}")
print(f"Nights dropped                  : {len(scored_sleep_df) - len(scored_sleep_filtered_df)}")

Total nights before flag filter : 3062
Total nights after flag filter  : 2585
Nights dropped                  : 477


In [14]:
## keep only actigraphy data from participants that did not withdraw from study 
valid_scored_df = scored_sleep_filtered_df[~scored_sleep_filtered_df['subject_id'].isin(pid_to_drop)].copy()

print(f"Total nights before flag filter : {len(scored_sleep_filtered_df)}")
print(f"Total nights after flag filter  : {len(valid_scored_df)}")
print(f"Nights dropped                  : {len(scored_sleep_filtered_df) - len(valid_scored_df)}")

Total nights before flag filter : 2585
Total nights after flag filter  : 2525
Nights dropped                  : 60


In [15]:
valid_scored_df['subject_id'].nunique()
## 249 unique participants
# scored_sleep_filtered_df['subject_id'].nunique()
# ## 258
# scored_sleep_df['subject_id'].nunique()
## 259

251

In [16]:
## Standardize datetimes and confirm they're in datetime format

valid_scored_df['start_datetime'] = pd.to_datetime(valid_scored_df['start_datetime'])
valid_scored_df['sleep_mid_dt']   = pd.to_datetime(valid_scored_df['sleep_mid_dt'])
valid_scored_df['sleep_onset_plus4h'] = pd.to_datetime(valid_scored_df['sleep_onset_plus4h'])
valid_scored_df['end_datetime'] = pd.to_datetime(valid_scored_df['end_datetime'])
## actigraphy_df pd.to_datime
actigraphy_df['date_time24']      = pd.to_datetime(actigraphy_df['date_time24'])

## doublecheck
print("Dtypes confirmed:")
print(valid_scored_df[['start_datetime', 'sleep_mid_dt', 'sleep_onset_plus4h', 'end_datetime']].dtypes)
print(actigraphy_df['date_time24'].dtype)

Dtypes confirmed:
start_datetime        datetime64[ns]
sleep_mid_dt          datetime64[ns]
sleep_onset_plus4h    datetime64[ns]
end_datetime          datetime64[ns]
dtype: object
datetime64[ns]


In [17]:
valid_scored_df.head()

,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-07-07 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-07-07 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-07-07 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-07-07 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00
5,DXA_001,DXA001,Sleep,6,Wednesday,2019-09-25,Wed,2026-07-07 23:31:00,2019-09-26,Thu,...,27.33,NaN,DXA,NaN,2019-09-25 23:31:00,2019-09-26 06:59:00,0.311111,448.0,2019-09-26 03:15:00,2019-09-26 03:31:00


## Calculate Fragmentation Index

Fragmentation Index = 

((Mobile Epochs + Immobile Bouts ≤1 min) / Total Immobile Bouts) * 100

In [18]:
## Fragmentation Index Calculation

## A = mobile epochs (activity count >= 40)
## B = sandwiched immobile epochs (activity < 40, but both pre- proceeding epochs >=40 ac)
## C = total immobile epochs (ac <40)

activity_threshold = 40

def compute_fragmentation_index(activity_series):
    """
    Compute Fragmentation index from series of activity counts > the sleep period (sleep start to sleep end)
    returns fragmentation index as a percentage, plus A, B, C, components
    fragmentation index = ((A+B)/C)*100
    """

    act = activity_series.reset_index(drop=True)

    mobile_mask = act >= activity_threshold
    A = int(mobile_mask.sum())

    immobile_mask = act < activity_threshold
    C = int(immobile_mask.sum())

    B = 0
    for i in range(1,len(act)-1):
        if (
            act[i] < activity_threshold and
            act[i-1] >= activity_threshold and
            act[i+1] >= activity_threshold
        ):
            B +=1

    fragmentation_index = round(((A+B)/C)*100, 2) if C >0 else None

    return {
        'A_mobile_epochs': A,
        'B_sandwiched_immobile': B,
        'C_total_immobile' : C,
        'fragmentation_index': fragmentation_index,
    }

## Test calculations

In [19]:
test_pid = ['DXA_001']
DXA_001_df = valid_scored_df[valid_scored_df['subject_id'].isin(test_pid)].copy()

print(DXA_001_df.shape)
display(DXA_001_df.head())

(13, 38)


,subject_id,passcode,interval_type,interval#,night,start_date,start_day,start_time,end_date,end_day,...,fragmentation,flag_actigraph,study,semester,start_datetime,end_datetime,time_diff,tst_min,sleep_mid_dt,sleep_onset_plus4h
0,DXA_001,DXA001,Sleep,1,Friday,2019-09-20,Fri,2026-07-07 23:21:00,2019-09-21,Sat,...,20.28,NaN,DXA,NaN,2019-09-20 23:21:00,2019-09-21 07:56:00,0.357639,515.0,2019-09-21 03:38:30,2019-09-21 03:21:00
2,DXA_001,DXA001,Sleep,3,Sunday,2019-09-22,Sun,2026-07-07 21:42:00,2019-09-23,Mon,...,20.75,NaN,DXA,NaN,2019-09-22 21:42:00,2019-09-23 08:30:00,0.450000,648.0,2019-09-23 03:06:00,2019-09-23 01:42:00
3,DXA_001,DXA001,Sleep,4,Monday,2019-09-23,Mon,2026-07-07 22:21:00,2019-09-24,Tue,...,28.54,NaN,DXA,NaN,2019-09-23 22:21:00,2019-09-24 07:49:00,0.394444,568.0,2019-09-24 03:05:00,2019-09-24 02:21:00
4,DXA_001,DXA001,Sleep,5,Tuesday,2019-09-25,Wed,2026-07-07 00:29:00,2019-09-25,Wed,...,17.80,NaN,DXA,NaN,2019-09-25 00:29:00,2019-09-25 08:20:00,0.327083,471.0,2019-09-25 04:24:30,2019-09-25 04:29:00
5,DXA_001,DXA001,Sleep,6,Wednesday,2019-09-25,Wed,2026-07-07 23:31:00,2019-09-26,Thu,...,27.33,NaN,DXA,NaN,2019-09-25 23:31:00,2019-09-26 06:59:00,0.311111,448.0,2019-09-26 03:15:00,2019-09-26 03:31:00


In [20]:
## testing DXA_001's records before the larger, full actigraphy dataset.
DXA_001_records = []

for _, row in DXA_001_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['end_datetime']      

    # slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    # --- sleep/wake classification ---
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    DXA_001_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

DXA_001_sleep_results_df = pd.DataFrame(DXA_001_records)

print(f"Results shape: {DXA_001_sleep_results_df.shape}")
DXA_001_sleep_results_df.head(15)

Results shape: (13, 13)


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24,3,491,5.50
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36,0,612,5.88
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43,5,525,9.14
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13,0,458,2.84
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28,1,420,6.90
5,DXA_001,10,2019-09-30 00:16:00,2019-09-30 08:06:00,470,438,32,32,93.19,25,1,445,5.84
6,DXA_001,11,2019-09-30 23:37:00,2019-10-01 07:18:00,461,429,32,32,93.06,28,0,433,6.47
7,DXA_001,12,2019-10-02 00:32:00,2019-10-02 07:59:00,447,414,33,33,92.62,22,0,425,5.18
8,DXA_001,13,2019-10-03 00:34:00,2019-10-03 06:29:00,355,333,22,22,93.80,19,1,336,5.95
9,DXA_001,14,2019-10-04 00:11:00,2019-10-04 07:21:00,430,411,19,19,95.58,18,0,412,4.37


#### diagnostics for verifying this worked properly

In [21]:
# # ── Diagnostic 1: confirm subject_id format matches in both dfs ───────────────
# print("=== scored df subject_id for DXA_001 ===")
# print(DXA_001_df['subject_id'].unique())

# print("\n=== actigraphy_df subject_id values (all unique) ===")
# print(actigraphy_df['subject_id'].unique())

# # direct check — does DXA_001's ID appear in actigraphy_df at all?
# scored_id = DXA_001_df['subject_id'].iloc[0]
# print(f"\nLooking for '{scored_id}' in actigraphy_df...")
# print(f"Match found: {scored_id in actigraphy_df['subject_id'].values}")

In [22]:
# # ── Diagnostic 2: check datetime ranges overlap ───────────────────────────────

# actig_001 = actigraphy_df[actigraphy_df['subject_id'] == scored_id]

# print(f"=== actigraphy_df epochs for {scored_id} ===")
# print(f"row count  : {len(actig_001):,}")
# print(f"date range : {actig_001['date_time24'].min()}  →  {actig_001['date_time24'].max()}")

# print(f"\n=== DXA_001_df sleep windows ===")
# print(DXA_001_df[['interval#', 'start_datetime', 'end_datetime']].to_string(index=False))

# # check each night individually
# print("\n=== epoch count per scored night ===")
# for _, row in DXA_001_df.iterrows():
#     mask = (
#         (actigraphy_df['subject_id']  == scored_id)          &
#         (actigraphy_df['date_time24'] >= row['start_datetime']) &
#         (actigraphy_df['date_time24'] <  row['end_datetime'])
#     )
#     n = mask.sum()
#     print(f"  interval# {row['interval#']}  →  {row['start_datetime']}  to  {row['end_datetime']}  →  {n} epochs")

In [23]:
# # ── Diagnostic 3: check interval_status values for DXA_001 epochs ────────────

# print(f"=== interval_status value counts for {scored_id} in actigraphy_df ===")
# print(actig_001['interval_status'].value_counts(dropna=False))

# print(f"\n=== sleep/wake value counts ===")
# print(actig_001['sleep/wake'].value_counts(dropna=False))

# print(f"\n=== sample of raw epochs ===")
# print(actig_001[['date_time24', 'interval_status', 'sleep/wake', 'activity']].head(10).to_string(index=False))

# Sleep Onset to Sleep_offset

In [24]:
valid_scored_df['subject_id'].nunique()

251

In [25]:
## Running nocturnal sleep metrics for full actigraphy dataset: valid_scored_df
full_sleep_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['end_datetime']      

    # slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    # --- sleep/wake classification ---
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    # WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    full_sleep_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

full_noc_sleep_results_df = pd.DataFrame(full_sleep_records)

print(f"Results shape: {full_noc_sleep_results_df.shape}")
full_noc_sleep_results_df.head(15)

Results shape: (2525, 13)


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 07:56:00,515,487,28,28,94.56,24.0,3.0,491.0,5.50
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 08:30:00,648,610,38,38,94.14,36.0,0.0,612.0,5.88
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 07:49:00,568,512,56,56,90.14,43.0,5.0,525.0,9.14
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 08:20:00,471,451,20,20,95.75,13.0,0.0,458.0,2.84
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 06:59:00,448,412,36,36,91.96,28.0,1.0,420.0,6.90
5,DXA_001,10,2019-09-30 00:16:00,2019-09-30 08:06:00,470,438,32,32,93.19,25.0,1.0,445.0,5.84
6,DXA_001,11,2019-09-30 23:37:00,2019-10-01 07:18:00,461,429,32,32,93.06,28.0,0.0,433.0,6.47
7,DXA_001,12,2019-10-02 00:32:00,2019-10-02 07:59:00,447,414,33,33,92.62,22.0,0.0,425.0,5.18
8,DXA_001,13,2019-10-03 00:34:00,2019-10-03 06:29:00,355,333,22,22,93.80,19.0,1.0,336.0,5.95
9,DXA_001,14,2019-10-04 00:11:00,2019-10-04 07:21:00,430,411,19,19,95.58,18.0,0.0,412.0,4.37


In [26]:
full_noc_sleep_results_df.tail()


,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
2520,SPU_154418,3,2024-03-28 01:07:00,2024-03-28 10:09:00,542,445,97,97,82.10,81.0,12.0,461.0,20.17
2521,SPU_154418,4,2024-03-28 23:31:00,2024-03-29 09:26:00,595,518,77,77,87.06,64.0,7.0,531.0,13.37
2522,SPU_154418,5,2024-03-30 01:10:00,2024-03-30 08:37:00,447,397,50,50,88.81,38.0,4.0,409.0,10.27
2523,SPU_154418,6,2024-03-30 22:02:00,2024-03-31 08:30:00,628,463,165,165,73.73,123.0,17.0,505.0,27.72
2524,SPU_154418,7,2024-04-01 00:50:00,2024-04-01 08:24:00,454,410,44,44,90.31,33.0,0.0,421.0,7.84


## Export Full_sleep_results


In [27]:
full_noc_sleep_results_df.to_excel('full_noc_sleep_results_df_070726.xlsx')

In [ ]:
asdfasfasfasdfasdf ## check in

# Sleep Metrics: sleep onset to mid-sleep

In [28]:
## all Sleep Records, to compare full nocturnal sleep with scored summary data (verification step)
onset_mid_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['sleep_mid_dt']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    ## WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    onset_mid_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

onset_mid_sleep_df = pd.DataFrame(onset_mid_records)

print(f"Results shape: {onset_mid_sleep_df.shape}")

Results shape: (2525, 13)


In [29]:
onset_mid_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 03:38:30,258,243,15,15,94.19,12.0,1.0,246.0,5.28
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 03:06:00,324,305,19,19,94.14,17.0,0.0,307.0,5.54
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 03:05:00,284,252,32,32,88.73,25.0,4.0,259.0,11.20
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 04:24:30,236,232,4,4,98.31,4.0,0.0,232.0,1.72
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 03:15:00,224,210,14,14,93.75,13.0,0.0,211.0,6.16


In [30]:
onset_mid_sleep_df.to_excel('Onset2mid_sleep_results_070726.xlsx')

# Sleep onset to sleep onset + 4h

In [31]:
## all Sleep Records, to compare full nocturnal sleep with scored summary data (verification step)
onset_plus4h_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['start_datetime']
    t_end   = row['sleep_mid_dt']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    ## WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    onset_plus4h_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

onset_plus4h_sleep_df = pd.DataFrame(onset_plus4h_records)

print(f"Results shape: {onset_plus4h_sleep_df.shape}")

Results shape: (2525, 13)


In [32]:
onset_plus4h_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-20 23:21:00,2019-09-21 03:38:30,258,243,15,15,94.19,12.0,1.0,246.0,5.28
1,DXA_001,3,2019-09-22 21:42:00,2019-09-23 03:06:00,324,305,19,19,94.14,17.0,0.0,307.0,5.54
2,DXA_001,4,2019-09-23 22:21:00,2019-09-24 03:05:00,284,252,32,32,88.73,25.0,4.0,259.0,11.20
3,DXA_001,5,2019-09-25 00:29:00,2019-09-25 04:24:30,236,232,4,4,98.31,4.0,0.0,232.0,1.72
4,DXA_001,6,2019-09-25 23:31:00,2019-09-26 03:15:00,224,210,14,14,93.75,13.0,0.0,211.0,6.16


In [33]:
onset_plus4h_sleep_df.to_excel('plus_results_df_070726.xlsx')

# Mid-sleep to Sleep offset

In [34]:
## all Sleep Records, to compare full nocturnal sleep with scored summary data (verification step)
second_half_records = []

for _, row in valid_scored_df.iterrows():
    pid     = row['subject_id']
    night   = row['interval#']
    t_start = row['sleep_mid_dt']
    t_end   = row['end_datetime']      

    ## slice epochs for this participant-night window
    mask = (
        (actigraphy_df['subject_id']  == pid)     &
        (actigraphy_df['date_time24'] >= t_start) &
        (actigraphy_df['date_time24'] <  t_end)
    )
    epochs = actigraphy_df.loc[mask, ['date_time24', 'interval_status',
                                       'sleep/wake', 'activity']].copy()

    window_min = len(epochs)

    ## sleep/wake classification
    true_sleep = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 0)
    micro_wake = (epochs['interval_status'] == 'REST-S') & (epochs['sleep/wake'] == 1)
    awake      = (epochs['sleep/wake'] == 1)

    TST_min = int(true_sleep.sum())

    ## WASO and fragmentation: only calculable after sleep onset
    first_sleep_pos = true_sleep.values.argmax() if true_sleep.any() else None

    if first_sleep_pos is not None and true_sleep.any():
        after_onset           = awake.iloc[first_sleep_pos + 1:]
        WASO_min              = int(after_onset.sum())
        sleep_period_activity = epochs['activity'].iloc[first_sleep_pos:].reset_index(drop=True)
        frag                  = compute_fragmentation_index(sleep_period_activity)
    else:
        WASO_min = 0
        frag     = {
            'A_mobile_epochs'      : None,
            'B_sandwiched_immobile': None,
            'C_total_immobile'     : None,
            'fragmentation_index'  : None,
        }

    efficiency = round(TST_min / window_min * 100, 2) if window_min > 0 else None

    second_half_records.append({
        'subject_id'      : pid,
        'interval#'       : night,
        'start_datetime'  : t_start,
        'end_datetime'    : t_end,
        'window_min'      : window_min,
        'TST_min'         : TST_min,
        'WASO_min'        : WASO_min,
        'micro_wake_min'  : int(micro_wake.sum()),
        'sleep_efficiency': efficiency,
        **frag,
    })

second_half_noc_sleep_df = pd.DataFrame(second_half_records)

print(f"Results shape: {second_half_noc_sleep_df.shape}")

Results shape: (2525, 13)


In [35]:
second_half_noc_sleep_df.head()

,subject_id,interval#,start_datetime,end_datetime,window_min,TST_min,WASO_min,micro_wake_min,sleep_efficiency,A_mobile_epochs,B_sandwiched_immobile,C_total_immobile,fragmentation_index
0,DXA_001,1,2019-09-21 03:38:30,2019-09-21 07:56:00,257,244,13,13,94.94,12.0,2.0,245.0,5.71
1,DXA_001,3,2019-09-23 03:06:00,2019-09-23 08:30:00,324,305,19,19,94.14,19.0,0.0,305.0,6.23
2,DXA_001,4,2019-09-24 03:05:00,2019-09-24 07:49:00,284,260,24,24,91.55,18.0,1.0,266.0,7.14
3,DXA_001,5,2019-09-25 04:24:30,2019-09-25 08:20:00,235,219,16,16,93.19,9.0,0.0,226.0,3.98
4,DXA_001,6,2019-09-26 03:15:00,2019-09-26 06:59:00,224,202,22,22,90.18,15.0,1.0,209.0,7.66


In [36]:
second_half_noc_sleep_df.to_excel('second_half_noc_sleep_df_070726.xlsx')

# Stop